# 01 - Exploratory Data Analysis

This notebook performs exploratory data analysis on the EncryptionGuard event data.
We examine event types, label distributions, timelines, and summary statistics.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configure plotting
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load events data
DATA_PATH = Path("../data/events.json")
with open(DATA_PATH) as f:
    events = json.load(f)

df = pd.DataFrame(events)

# Print basic stats
print(f"Total events: {len(df)}")
print(f"\nEvent types ({df['event_type'].nunique()}):")
print(df["event_type"].value_counts())
print(f"\nLabels ({df['label'].nunique() if 'label' in df.columns else 'N/A'}):")
if "label" in df.columns:
    print(df["label"].value_counts())
else:
    print("No 'label' column found.")

df.head()

## Event Type & Label Distribution

Bar charts showing the distribution of event types and labels in the dataset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Event type distribution
event_counts = df["event_type"].value_counts()
sns.barplot(x=event_counts.values, y=event_counts.index, ax=axes[0], palette="viridis")
axes[0].set_title("Distribution of Event Types")
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Event Type")

# Label distribution
if "label" in df.columns:
    label_counts = df["label"].value_counts()
    sns.barplot(x=label_counts.values, y=label_counts.index, ax=axes[1], palette="magma")
    axes[1].set_title("Distribution of Labels")
    axes[1].set_xlabel("Count")
    axes[1].set_ylabel("Label")
else:
    axes[1].text(0.5, 0.5, "No label column", ha="center", va="center")
    axes[1].set_title("Label Distribution (N/A)")

plt.tight_layout()
plt.show()

## Event Timeline

Visualizing the temporal distribution of events over time.

In [ ]:
# Parse timestamps
time_col = None
for col in ["timestamp", "created_at", "received_at", "event_time"]:
    if col in df.columns:
        time_col = col
        break

if time_col:
    df["parsed_time"] = pd.to_datetime(df[time_col])
    df["date"] = df["parsed_time"].dt.date

    daily_counts = df.groupby("date").size()

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(daily_counts.index, daily_counts.values, marker="o", linewidth=1.5)
    ax.fill_between(daily_counts.index, daily_counts.values, alpha=0.3)
    ax.set_title("Events Over Time (Daily)")
    ax.set_xlabel("Date")
    ax.set_ylabel("Number of Events")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No timestamp column found. Available columns:", list(df.columns))

## Summary Statistics

Descriptive statistics for numeric columns and overall data quality overview.

In [ ]:
# Summary statistics for numeric columns
print("=== Numeric Column Statistics ===")
print(df.describe())

print("\n=== Data Quality ===")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"\nMissing values per column:")
print(df.isnull().sum())

print(f"\nColumn dtypes:")
print(df.dtypes)